# IMDB Movie Review Sentiment with LSTM

## Objectives
Binary **sentiment classification** (positive/negative) on text using LSTM.

## RNN Theory
**RNN** maintains a hidden state across time steps: $h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b)$  
**LSTM** adds gates (forget, input, output) to capture long-range dependencies and mitigate vanishing gradients.


In [ ]:
# Optional: install dependencies (uncomment if needed)
# !pip install -q numpy pandas matplotlib seaborn scikit-learn tensorflow requests yfinance

import warnings
warnings.filterwarnings("ignore")

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error, r2_score,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid")
print("TensorFlow:", tf.__version__)


## Business Context
Retailers and banks analyze reviews, complaints, and chat logs for sentiment routing.


In [ ]:
max_features = 10000
max_len = 200
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=max_features)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.15, random_state=SEED)


In [ ]:
# EDA — sequence lengths
train_lens = [len(s) for s in x_train[:5000]]
sns.histplot(train_lens, bins=30)
plt.title("Review sequence lengths (subset)")
plt.xlabel("Tokens")
plt.show()


In [ ]:
# Pad sequences to fixed length
x_train_p = keras.preprocessing.sequence.pad_sequences(x_train, maxlen=max_len)
x_val_p = keras.preprocessing.sequence.pad_sequences(x_val, maxlen=max_len)
x_test_p = keras.preprocessing.sequence.pad_sequences(x_test, maxlen=max_len)


In [ ]:
model = models.Sequential([
    layers.Embedding(max_features, 128, input_length=max_len),
    layers.LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy", "AUC"])
model.summary()


In [ ]:
rnn_cb = [
    callbacks.ModelCheckpoint("lstm_imdb_best.keras", save_best_only=True, monitor="val_accuracy", mode="max"),
    callbacks.EarlyStopping(patience=3, restore_best_weights=True, monitor="val_accuracy", mode="max"),
]
history = model.fit(
    x_train_p, y_train,
    validation_data=(x_val_p, y_val),
    epochs=5,
    batch_size=128,
    callbacks=rnn_cb,
)


In [ ]:
test_loss, test_acc, test_auc = model.evaluate(x_test_p, y_test, verbose=0)
print(f"Test acc: {test_acc:.4f}")
y_prob = model.predict(x_test_p[:500], verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)
print(classification_report(y_test[:500], y_pred))


In [ ]:
# Inference demo
sample = x_test_p[:3]
probs = model.predict(sample, verbose=0)
for i, p in enumerate(probs):
    print(f"Review {i}: sentiment={'positive' if p[0]>0.5 else 'negative'} ({p[0]:.3f})")
model.save("lstm_imdb_final.keras")


## Deployment Notes

1. **Serving**: Export with `model.export("saved_model")` for TensorFlow Serving, or wrap `predict` in FastAPI/Flask.
2. **Preprocessing**: Always apply the **same** scaler/encoder fitted on training data (`scaler.pkl`).
3. **Monitoring**: Track input drift, latency, and prediction distribution on live traffic.
4. **Retraining**: Schedule periodic retrain when performance drops below SLA.
5. **Security**: Do not log PII; use HTTPS and auth on inference endpoints.
